# 05 ? SQLite Relational Data Layer

## Objective
Create a reproducible local SQLite database with one source-faithful table for each EM2023 CSV module. Preserve the source headers and every value as text, including leading zeros, whitespace, empty strings, and the empty vehicle header. The raw CSV files remain the immutable source of record.

## Relational design
- Technical primary keys use the empirically unique fields `key_hg`, `cod_vh`, `key_persona`, `key_viaje`, and `key_etapa`. Observed alternate identifiers remain in their source columns.
- Foreign keys use the paired identifier mappings checked across the source modules: household code/key, person code/key, and trip code/key. These checks establish observed coverage, not official variable definitions.
- All source fields are declared `TEXT`. No value conversion, header normalization, filtering, weighting, OD aggregation, or mode-choice dataset construction occurs here.

## Outputs and safeguards
The notebook creates `data/relational/em2023_relational.sqlite`, which is excluded from Git, plus aggregate table and relationship reports in `docs/data_dictionary/`. The database is built in a temporary file and replaces an existing copy only after row counts, source headers, foreign keys, and SQLite integrity pass.

In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate pyproject.toml from the current directory.")
src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from mode_choice.relational_database import build_relational_database, write_relational_database_reports

database_path, table_results, relationship_results = build_relational_database(PROJECT_ROOT)
table_report, relationship_report = write_relational_database_reports(
    PROJECT_ROOT, table_results, relationship_results
)
tables = pd.DataFrame(table_results)
relationships = pd.DataFrame(relationship_results)
print(f"SQLite database: {database_path.relative_to(PROJECT_ROOT)}")
print(f"Table report: {table_report.relative_to(PROJECT_ROOT)}")
print(f"Relationship report: {relationship_report.relative_to(PROJECT_ROOT)}")
display(tables)
display(relationships)
assert tables["source_row_count"].equals(tables["loaded_row_count"]), "A source table row count changed during loading."
assert tables["source_headers_preserved"].all(), "A source header was not preserved."
assert relationships["foreign_key_violation_count"].eq(0).all(), "A relational constraint failed."

SQLite database: data\relational\em2023_relational.sqlite
Table report: docs\data_dictionary\relational_database_tables.csv
Relationship report: docs\data_dictionary\relational_database_relationships.csv


,table_name,source_path,source_row_count,loaded_row_count,source_column_count,source_headers_preserved,stored_column_type,primary_key_field,alternate_unique_fields_json,row_width_mismatch_count,status
0,hogares,data/sources/raw/Database/a. Modulo hogares.csv,22755,22755,27,True,TEXT,key_hg,"[""cod_hog""]",0,loaded
1,vehiculos,data/sources/raw/Database/b. Modulo vehiculos.csv,17392,17392,19,True,TEXT,cod_vh,[],0,loaded
2,personas,data/sources/raw/Database/c. Modulo personas.csv,67556,67556,40,True,TEXT,key_persona,"[""cod_per""]",0,loaded
3,viajes,data/sources/raw/Database/d. Modulo viajes.csv,100174,100174,47,True,TEXT,key_viaje,"[""cod_vj""]",0,loaded
4,etapas,data/sources/raw/Database/e. Modulo etapas.csv,110881,110881,38,True,TEXT,key_etapa,"[""cod_eta""]",0,loaded


,child_table,child_fields_json,parent_table,parent_fields_json,child_row_count,foreign_key_violation_count,validation_status,status_note
0,vehiculos,"[""cod_hog"", ""key_hg""]",hogares,"[""cod_hog"", ""key_hg""]",17392,0,empirically_validated,Exact values and paired identifiers were check...
1,personas,"[""cod_hg"", ""key_hg""]",hogares,"[""cod_hog"", ""key_hg""]",67556,0,empirically_validated,Exact values and paired identifiers were check...
2,viajes,"[""cod_hg"", ""key_hg""]",hogares,"[""cod_hog"", ""key_hg""]",100174,0,empirically_validated,Exact values and paired identifiers were check...
3,viajes,"[""cod_pers"", ""key_pers""]",personas,"[""cod_per"", ""key_persona""]",100174,0,empirically_validated,Exact values and paired identifiers were check...
4,etapas,"[""cod_vj"", ""key_viaje""]",viajes,"[""cod_vj"", ""key_viaje""]",110881,0,empirically_validated,Exact values and paired identifiers were check...
5,etapas,"[""cod_pers""]",personas,"[""cod_per""]",110881,0,empirically_validated,Exact values and paired identifiers were check...
